# MEV — Máxima Extração de Valor na Rede Polygon
**Pesquisa FAPEMIG** | Coordenador: Prof. José Augusto Miranda Nacif | Bolsista: Aline Cristina Santos Silva

---

Este notebook documenta a **coleta e análise de dados on-chain** da rede Polygon (Layer-2),
com foco na medição do *Reordering Slippage* em swaps da Uniswap V3.

### Objetivo
Coletar eventos de Swap reais da blockchain para, posteriormente, calcular o slippage de reordenação
e comparar com a linha de base da Ethereum Mainnet — validando a hipótese **H** da proposta:
> *Contratos inteligentes em redes L2 apresentam um reordering slippage significativamente menor do que na rede principal Ethereum.*

### Perguntas de Pesquisa endereçadas
- **RQ1:** Qual a magnitude da diferença no Reordering Slippage médio entre Ethereum Mainnet e Polygon?
- **RQ2:** Qual a proporção entre Slippage Adversário (MEV) e Slippage de Colisão (benigno)?
- **RQ3:** Ativos voláteis (memecoins) apresentam maior slippage adversário também em L2?

---

## 1. Instalação de Dependências

Bibliotecas necessárias:
- **`web3`** — interface com nós da blockchain via RPC
- **`pandas`** — manipulação e análise dos dados coletados
- **`python-dotenv`** — carrega variáveis de ambiente do arquivo `.env` (protege a API key)
- **`matplotlib` / `seaborn`** — visualizações

In [14]:
# Execute uma vez para instalar as dependências
%pip install web3 pandas python-dotenv matplotlib seaborn --quiet

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/pip/__main__.py", line 22, in <module>
    from pip._internal.cli.main import main as _main
  File "/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 10, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/pip/_internal/build_env.py", line 19, in <module>
    from pi

## 2. Configuração — Carregando Credenciais

A API key da Alchemy fica armazenada no arquivo **`.env`** (nunca versionado no GitHub).
O arquivo `.env.example` no repositório documenta quais variáveis são necessárias sem expor os valores reais.



## 3. Conexão com a Rede Polygon via Alchemy

A conexão é feita via **RPC (Remote Procedure Call)** — um protocolo que permite consultar
o estado da blockchain sem precisar baixar todos os dados localmente.

A **Alchemy** atua como provedor de nó, fornecendo acesso à Polygon Mainnet de forma confiável e escalável.

**Por que Polygon?**
Por ser uma rede L2, espera-se que o *reordering slippage* seja menor do que na Ethereum Mainnet —
essa é exatamente a hipótese que esta pesquisa busca validar empiricamente.

In [3]:
import os
import requests
from dotenv import load_dotenv
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

load_dotenv(override=True)
API_KEY = os.getenv("ALCHEMY_API_KEY")
RPC_URL = f"https://polygon-mainnet.g.alchemy.com/v2/{API_KEY}"

# Sessão customizada sem verificação SSL
session = requests.Session()
session.verify = False

from web3.middleware import ExtraDataToPOAMiddleware
from requests.adapters import HTTPAdapter

w3 = Web3(Web3.HTTPProvider(RPC_URL, session=session))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)

try:
    bloco = w3.eth.block_number
    print(f" Conectado! Bloco: {bloco:,}")
except Exception as e:
    print(f" Erro: {e}")

 Conectado! Bloco: 87,925,318


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 4. Definição do Contrato — Pool Uniswap V3 (USDC/WETH)

Na Uniswap V3, cada par de tokens possui um **contrato de pool dedicado**.
Toda vez que um swap ocorre, o contrato emite um **evento `Swap`** — um log público e imutável
gravado na blockchain, contendo:

| Campo | Descrição | Relevância para a pesquisa |
|---|---|---|
| `sqrtPriceX96` | Preço de execução real codificado | Base para calcular o slippage |
| `amount0 / amount1` | Volume negociado de cada token | Tamanho do swap (impacto no preço) |
| `sender / recipient` | Endereços envolvidos | Identificar padrões de ataque sandwich |
| `tick` | Posição na curva de preço | Faixa de liquidez utilizada |
| `blockNumber` | Bloco em que ocorreu | Agrupar transações por bloco para cálculo do reordering slippage |

Começamos com o pool **USDC/WETH 0.05%** por ser um dos mais líquidos na Polygon —
ideal para calibrar a metodologia antes de analisar ativos voláteis (RQ3).

In [ ]:
# Endereço do pool USDC/WETH 0.05% na Polygon (Uniswap V3)
POOL_ADDRESS = Web3.to_checksum_address("0x45dda9cb7c25131df268515131f647d726f50608")

# ABI mínimo — apenas o evento Swap (não precisamos do ABI completo)
POOL_ABI = [
    {
        "anonymous": False,
        "inputs": [
            {"indexed": True,  "name": "sender",       "type": "address"},
            {"indexed": True,  "name": "recipient",    "type": "address"},
            {"indexed": False, "name": "amount0",      "type": "int256"},
            {"indexed": False, "name": "amount1",      "type": "int256"},
            {"indexed": False, "name": "sqrtPriceX96", "type": "uint160"},
            {"indexed": False, "name": "liquidity",    "type": "uint128"},
            {"indexed": False, "name": "tick",         "type": "int24"}
        ],
        "name": "Swap",
        "type": "event"
    }
]

contrato = w3.eth.contract(address=POOL_ADDRESS, abi=POOL_ABI)
print(f" Contrato carregado: {POOL_ADDRESS}")
print(f" Pool: USDC/WETH 0.05% — Uniswap V3 na Polygon")

 Contrato carregado: 0xA374094527e1673A86dE625aa59517c5dE346d32
 Pool: USDC/WETH 0.05% — Uniswap V3 na Polygon


## 5. Coleta de Eventos de Swap

Estamos usando o pool WMATIC/USDC ( o mais negociado da rede) e estamos usando a janela para 3.000 blocos. O resultado foi 441 swaps cim 53 blocos tendo 2 ou mais swaps, que é exatamente o mínimo que precisamos para calcular o Reordering Slippage. 

In [ ]:
import time
import pandas as pd
from pathlib import Path

POOL_ADDRESS_WMATIC = Web3.to_checksum_address("0xa374094527e1673a86de625aa59517c5de346d32")
contrato_wmatic = w3.eth.contract(address=POOL_ADDRESS_WMATIC, abi=POOL_ABI)

JANELA_BLOCOS = 10
TOTAL_BLOCOS  = 3000

# ── Salva/reutiliza sempre o mesmo bloco_fim ──────────────────────────────────
ARQUIVO_BLOCO = Path("dataFrame/bloco_fim.txt")

if ARQUIVO_BLOCO.exists():
    bloco_fim = int(ARQUIVO_BLOCO.read_text().strip())
    print(f"📂 Reutilizando bloco_fim salvo: {bloco_fim:,}")
else:
    bloco_fim = w3.eth.block_number
    ARQUIVO_BLOCO.parent.mkdir(parents=True, exist_ok=True)
    ARQUIVO_BLOCO.write_text(str(bloco_fim))
    print(f"💾 Novo bloco_fim salvo: {bloco_fim:,}")

bloco_inicio = bloco_fim - TOTAL_BLOCOS

print(f"Coletando {TOTAL_BLOCOS:,} blocos em lotes de {JANELA_BLOCOS}...")
print(f"Estimativa: ~{TOTAL_BLOCOS // JANELA_BLOCOS * 0.3 / 60:.0f} minutos")
print(f"Bloco {bloco_inicio:,} → {bloco_fim:,}")
print()

todos_eventos = []
erros = 0

for i, inicio in enumerate(range(bloco_inicio, bloco_fim, JANELA_BLOCOS)):
    fim = min(inicio + JANELA_BLOCOS - 1, bloco_fim)
    try:
        eventos = contrato_wmatic.events.Swap.get_logs(
            from_block=inicio,
            to_block=fim
        )
        todos_eventos.extend(eventos)
        if i % 50 == 0:
            print(f"  Progresso: bloco {inicio:,} | {len(todos_eventos)} swaps acumulados")
    except Exception as e:
        erros += 1
    time.sleep(0.3)

print(f"\nTotal coletado : {len(todos_eventos)} swaps")
print(f"Erros          : {erros}")

# ── Inspeciona um evento completo ─────────────────────────────────────────────
if todos_eventos:
    print("\n── Exemplo de evento bruto (1º swap coletado) ──")
    ev = todos_eventos[0]
    print(f"  Campos do topo : {list(ev.keys())}")
    print(f"  Args (campos do Swap):")
    for campo, valor in ev["args"].items():
        print(f"    {campo:20s} = {valor}")

# ── Diagnóstico ───────────────────────────────────────────────────────────────
if todos_eventos:
    _tmp = pd.DataFrame([{"bloco": e["blockNumber"]} for e in todos_eventos])
    _por_bloco = _tmp.groupby("bloco").size()
    uteis = (_por_bloco >= 2).sum()
    print(f"\nBlocos com 2+ swaps : {uteis}")
    print(f"Média por bloco     : {_por_bloco.mean():.2f}")
    print(f"Máximo num bloco    : {_por_bloco.max()}")

    if uteis < 30:
        print("\n⚠ Ainda poucos blocos úteis — considere aumentar TOTAL_BLOCOS para 5000")
    else:
        print("\n✓ Amostra suficiente para calcular o Reordering Slippage!")

 Reutilizando bloco_fim salvo: 87,925,372
Coletando 5,000 blocos em lotes de 10...
Bloco 87,920,372 → 87,925,372



/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  Progresso: bloco 87,920,372 | 2 swaps acumulados


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

  Progresso: bloco 87,920,872 | 44 swaps acumulados


...

  Progresso: bloco 87,924,372 | 295 swaps acumulados


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

  Progresso: bloco 87,924,872 | 352 swaps acumulados


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv


Total coletado : 407 swaps
Erros          : 0

── Exemplo de evento bruto (1º swap coletado) ──
  Campos do topo : ['args', 'event', 'logIndex', 'transactionIndex', 'transactionHash', 'address', 'blockHash', 'blockNumber']
  Args (campos do Swap):
    sender               = 0x7150ea07D00d8E5a46bcC809f1c9FDf5cb5f8E81
    recipient            = 0x7150ea07D00d8E5a46bcC809f1c9FDf5cb5f8E81
    amount0              = -9225957192452658138
    amount1              = 836030
    sqrtPriceX96         = 23843915710341785920512
    liquidity            = 368185353887356212
    tick                 = -300342

Blocos com 2+ swaps : 35
Média por bloco     : 1.12
Máximo num bloco    : 5


## 6. Estruturação dos Dados em DataFrame

Convertemos os eventos brutos da blockchain em um **DataFrame pandas** estruturado.
Cada linha representa um swap individual, com todos os campos necessários para
o cálculo do Reordering Slippage na próxima etapa.

In [11]:
import csv
import json
from pathlib import Path

def evento_para_dict(evento):
    """Converte um evento Web3 para um dicionário plano com todos os campos."""
    d = {}

    # ── Campos do topo ────────────────────────────────────────────────────────
    d["event"]            = evento.get("event")
    d["address"]          = evento.get("address")
    d["blockNumber"]      = evento.get("blockNumber")
    d["transactionIndex"] = evento.get("transactionIndex")
    d["logIndex"]         = evento.get("logIndex")

    # HexBytes → string legível
    tx_hash    = evento.get("transactionHash")
    block_hash = evento.get("blockHash")
    d["transactionHash"] = tx_hash.hex()    if tx_hash    else None
    d["blockHash"]       = block_hash.hex() if block_hash else None

    # ── Args (parâmetros do Swap) ─────────────────────────────────────────────
    args = evento.get("args", {})
    for chave, valor in args.items():
        if hasattr(valor, "hex"):                  # HexBytes
            valor = valor.hex()
        elif isinstance(valor, (dict, list)):       # estrutura aninhada
            valor = json.dumps(valor, default=str)
        d[f"args_{chave}"] = valor

    return d


# ── Gera as linhas ────────────────────────────────────────────────────────────
linhas = [evento_para_dict(e) for e in todos_eventos]

if not linhas:
    print("Nenhum evento para salvar.")
else:
    todas_colunas = list(dict.fromkeys(k for linha in linhas for k in linha))

    caminho_csv = Path("dataFrame/swaps.csv")
    with caminho_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=todas_colunas, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(linhas)

    print(f"{len(linhas)} swaps salvos em '{caminho_csv.resolve()}'")
    print(f"   Colunas ({len(todas_colunas)}): {todas_colunas}")

407 swaps salvos em '/home/aline/slippage-analysis/dataFrame/swaps.csv'
   Colunas (14): ['event', 'address', 'blockNumber', 'transactionIndex', 'logIndex', 'transactionHash', 'blockHash', 'args_sender', 'args_recipient', 'args_amount0', 'args_amount1', 'args_sqrtPriceX96', 'args_liquidity', 'args_tick']


## 7. Conversão do Preço (sqrtPriceX96 → Preço Legível)

O campo `sqrtPriceX96` armazena o preço em formato interno da Uniswap V3:
a **raiz quadrada do preço**, multiplicada por 2⁹⁶ (para evitar decimais na EVM).

Para obter o preço real USDC por WMATIC , aplicamos:

$$price = \left(\frac{sqrtPriceX96}{2^{96}}\right)^2 \times \frac{10^{decimais\_token0}}{10^{decimais\_token1}}$$

Para USDC (6 decimais) / WMATIC (18 decimais):

$$price_{USDC/WMATIC} = \left(\frac{sqrtPriceX96}{2^{96}}\right)^2 \times 10^{12}$$

 Este preço de execução real é a base para calcular o **Reordering Slippage** — a diferença entre o preço que o usuário obteve e o preço que obteria em uma ordem aleatória de transações.

In [12]:
import pandas as pd

# ── Carrega os dados do novo pool (WMATIC/USDC) ───────────────────────────────
df = pd.read_csv("dataFrame/swaps.csv")

df = df.rename(columns={
    "blockNumber":       "bloco",
    "transactionIndex":  "tx_index",
    "args_sqrtPriceX96": "sqrtPriceX96",
    "args_amount0":      "WMATIC",
    "args_amount1":      "USDC",
    "args_sender":       "sender",
    "args_recipient":    "recipient",
})

df["sqrtPriceX96"] = pd.to_numeric(df["sqrtPriceX96"])
df["amount0"]      = pd.to_numeric(df["WMATIC"])
df["amount1"]      = pd.to_numeric(df["USDC"])

# ── Conversão de preço — pool WMATIC/USDC ────────────────────────────────────
# token0 = WMATIC (18 decimais)
# token1 = USDC   (6 decimais)
# sqrtPriceX96 dá sqrt(token1/token0) em unidades brutas
# Ajuste decimal: 10^(decimais_token1 - decimais_token0) = 10^(6-18) = 10^-12

Q96 = 2 ** 96

def sqrt_price_to_price_wmatic(sqrt_price_x96):
    """
    Converte sqrtPriceX96 → preço USDC por WMATIC.
    
    Passo a passo:
    1. Divide por 2^96 para desfazer o encoding da Uniswap
    2. Eleva ao quadrado para desfazer a raiz quadrada
    3. Multiplica por 10^(6-18) para corrigir as casas decimais
       (USDC tem 6, WMATIC tem 18 — diferença de 12 casas)
    """
    preco_bruto = (sqrt_price_x96 / Q96) ** 2   # ainda em unidades brutas
    return preco_bruto * 1e12                     # corrige os 12 decimais

df["preco_execucao"] = df["sqrtPriceX96"].apply(sqrt_price_to_price_wmatic)

# ── Resultado ─────────────────────────────────────────────────────────────────
print("Preços calculados para o pool WMATIC/USDC!")
print(f"  Preço médio : ${df['preco_execucao'].mean():.4f} USDC/WMATIC")
print(f"  Mínimo      : ${df['preco_execucao'].min():.4f}")
print(f"  Máximo      : ${df['preco_execucao'].max():.4f}")
print(f"  Total swaps : {len(df)}")
print()

# ── Diagnóstico dos blocos úteis ──────────────────────────────────────────────
swaps_por_bloco = df.groupby("bloco").size()
blocos_uteis    = swaps_por_bloco[swaps_por_bloco >= 2]

print(f"Blocos com 2+ swaps : {len(blocos_uteis)}")
print(f"Swaps nesses blocos : {blocos_uteis.sum()}")
print()

# ── Tabela dos primeiros swaps ────────────────────────────────────────────────
df[["bloco", "tx_index", "preco_execucao", "WMATIC", "USDC"]].head(10)

Preços calculados para o pool WMATIC/USDC!
  Preço médio : $0.0913 USDC/WMATIC
  Mínimo      : $0.0906
  Máximo      : $0.0920
  Total swaps : 407

Blocos com 2+ swaps : 35
Swaps nesses blocos : 80



,bloco,tx_index,preco_execucao,WMATIC,USDC
0,87920373,203,0.090572,-9225957192452658138,836030
1,87920374,8,0.090708,-914991660841360995990,82976529
2,87920388,68,0.090711,-18095423785520712043,1642247
3,87920415,9,0.090853,-958102088208500866682,87021830
4,87920417,5,0.090942,-596518348199562468700,54249143
5,87920418,151,0.090930,80525500000000000000,-7318997
6,87920421,119,0.090922,51196255099705594364,-4652748
7,87920422,119,0.090884,256997085176322654208,-23350167
8,87920423,59,0.090835,329262615467199570473,-29901698
9,87920424,98,0.090818,117867554776292009313,-10700124


Quando WMATIC > 0 e USDC < 0 significa que o pool recebeu WMATIC e entregou USDC. Quando WMATIC<0 e USDC>0 significa que o pool recebeu USDC e entregou WMATIC. 

## 8. Cálculo do Reordering Slippage (Adams et al., 2023)

O **Reordering Slippage** compara o preço real de execução de um swap com o preço que ele teria obtido caso os trades do mesmo bloco fossem ordenados aleatoriamente. Se o preço real for pior do que a média das ordens aleatórias, há evidência de reordenação adversarial (MEV/sandwich).

### Fórmula (Definição 3.2)

$$\text{reorderingSlippage}_i = \left(\frac{\text{realizedPrice}_i}{\mathbb{E}_{\pi}[\text{hypotheticalPrice}_i(\pi(S))]} - 1\right) \times -10000$$

onde:
- $\text{realizedPrice}_i$ — preço real de execução do swap $i$
- $\pi(S)$ — permutação aleatória dos trades do bloco $B$
- $\mathbb{E}_{\pi}[\cdot]$ — média sobre `N_PERMS = 200` permutações amostradas

### Simulação do preço hipotético

Como não temos o estado interno do pool a cada bloco, aproximamos o AMM pela invariante $xy = k$: partimos de uma reserva inicial estimada e propagamos os trades anteriores a $i$ (na nova ordem) para descobrir qual seria o preço de execução de $i$ nessa ordenação hipotética.

### Interpretação

| Reordering Slippage | Classificação | Interpretação |
|---|---|---|
| `RS < 0` | **Adversarial** | Usuário pagou mais do que numa ordem aleatória — suspeito de MEV |
| `RS ≥ 0` | **Benigno** | Collision slippage ou price improvement |

In [13]:
import numpy as np
import pandas as pd
from itertools import permutations

# ── Parâmetros ────────────────────────────────────────────────────────────────
N_PERMS = 200          # número de permutações amostradas por bloco
SEED    = 42           # reprodutibilidade
rng     = np.random.default_rng(SEED)

# ── Função auxiliar: simula o preço de execução de um trade em uma ordenação ──

def simular_preco_hipotetico(df_bloco: pd.DataFrame, idx_alvo: int, ordem: list) -> float:
    """
    Simula o preço de execução do trade `idx_alvo` dado que os trades do bloco
    são executados na `ordem` fornecida, partindo do estado inicial do pool
    estimado pelo primeiro trade do bloco.

    Aproximação via AMM xy=k (sem liquidez concentrada).

    Parâmetros
    ----------
    df_bloco : DataFrame com colunas amount0 (WMATIC) e amount1 (USDC)
    idx_alvo : posição LOCAL (0..n-1) do trade de interesse em df_bloco
    ordem    : lista com a permutação das posições locais

    Retorna
    -------
    Preço hipotético em USDC/WMATIC
    """
    trades = df_bloco[["amount0", "amount1"]].values  # shape (n, 2)

    # Estima o estado INICIAL do pool antes deste bloco:
    # Usamos os amounts do primeiro trade como referência de escala.
    # Ponto de partida: reservas tais que o preço inicial seja o
    # preco_execucao do primeiro trade do bloco.
    preco_inicial = df_bloco["preco_execucao"].iloc[0]

    # Reserva inicial sintética: x0 (WMATIC), y0 (USDC) com xy = k
    # Escolhemos x0 = |sum amount0| * 10 para ter profundidade razoável
    x0 = abs(trades[:, 0]).sum() * 10   # reserva WMATIC inicial (unidades brutas)
    y0 = x0 * preco_inicial             # reserva USDC inicial
    k  = x0 * y0                        # invariante

    x, y = x0, y0

    # Executa os trades anteriores ao alvo (na ordem hipotética)
    pos_alvo_na_nova_ordem = ordem.index(idx_alvo)

    for pos in ordem[:pos_alvo_na_nova_ordem]:
        delta0 = trades[pos, 0]   # amount0 (WMATIC): > 0 se pool recebeu WMATIC
        x_novo = x + delta0
        if x_novo <= 0:
            x_novo = x * 0.001    # proteção numérica
        y = k / x_novo
        x = x_novo

    # Agora executa o trade alvo e mede o preço resultante
    delta0_alvo = trades[idx_alvo, 0]
    x_novo = x + delta0_alvo
    if x_novo <= 0:
        x_novo = x * 0.001

    y_novo = k / x_novo

    delta1_hipotetico = y - y_novo     # USDC que sai do pool (positivo = saiu USDC)

    # Preço = USDC movimentado / WMATIC movimentado (valor absoluto)
    denom = abs(delta0_alvo)
    if denom == 0:
        return np.nan
    return abs(delta1_hipotetico) / denom


# ── Cálculo principal ─────────────────────────────────────────────────────────

resultados = []

blocos_com_multiplos = df.groupby("bloco").filter(lambda g: len(g) >= 2)["bloco"].unique()

print(f"Calculando reordering slippage para {len(blocos_com_multiplos)} blocos...")
print(f"Permutações por bloco: {N_PERMS}\n")

for i_bloco, bloco_id in enumerate(blocos_com_multiplos):
    df_bloco = df[df["bloco"] == bloco_id].reset_index(drop=True)
    n = len(df_bloco)

    # Para cada trade no bloco, amostramos N_PERMS permutações aleatórias
    for idx_alvo in range(n):
        precos_hipoteticos = []

        for _ in range(N_PERMS):
            ordem = list(rng.permutation(n))
            ph = simular_preco_hipotetico(df_bloco, idx_alvo, ordem)
            if ph is not None and not np.isnan(ph) and ph > 0:
                precos_hipoteticos.append(ph)

        if not precos_hipoteticos:
            continue

        preco_realizado   = df_bloco["preco_execucao"].iloc[idx_alvo]
        preco_hipot_medio = np.mean(precos_hipoteticos)

        # Fórmula do artigo (Adams et al. 2023, Def. 3.2)
        # reorderingSlippage_i = (realizedPrice_i / E[hypotheticalPrice_i] − 1) × −10000
        rs = (preco_realizado / preco_hipot_medio - 1) * -10000

        resultados.append({
            "bloco":               bloco_id,
            "tx_index":            df_bloco["tx_index"].iloc[idx_alvo],
            "preco_realizado":     preco_realizado,
            "preco_hipotetico_medio": preco_hipot_medio,
            "reordering_slippage_bps": rs,
            "n_swaps_no_bloco":    n,
        })

    # Progresso
    if (i_bloco + 1) % 10 == 0:
        print(f"  {i_bloco + 1}/{len(blocos_com_multiplos)} blocos processados...")

df_rs = pd.DataFrame(resultados)

# ── Resultados ────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"REORDERING SLIPPAGE — Pool WMATIC/USDC — Polygon")
print(f"{'='*55}")
print(f"Swaps analisados          : {len(df_rs):,}")
print(f"Blocos analisados         : {df_rs['bloco'].nunique():,}")
print()
print(f"Reordering Slippage (bps):")
print(f"  Média                   : {df_rs['reordering_slippage_bps'].mean():.4f}")
print(f"  Mediana                 : {df_rs['reordering_slippage_bps'].median():.4f}")
print(f"  Desvio padrão           : {df_rs['reordering_slippage_bps'].std():.4f}")
print(f"  Mín                     : {df_rs['reordering_slippage_bps'].min():.4f}")
print(f"  Máx                     : {df_rs['reordering_slippage_bps'].max():.4f}")
print()

# Adversarial (RS < 0: usuário pagou mais do que numa ordem aleatória)
adversarial = df_rs[df_rs["reordering_slippage_bps"] < 0]
benign      = df_rs[df_rs["reordering_slippage_bps"] >= 0]
print(f"Adversarial (RS < 0)      : {len(adversarial):,}  ({len(adversarial)/len(df_rs)*100:.1f}%)")
print(f"Benigno    (RS >= 0)      : {len(benign):,}  ({len(benign)/len(df_rs)*100:.1f}%)")
print()
print(f"RS médio adversarial      : {adversarial['reordering_slippage_bps'].mean():.4f} bps")
print(f"RS médio benigno          : {benign['reordering_slippage_bps'].mean():.4f} bps")

# ── Salva resultado ───────────────────────────────────────────────────────────
df_rs.to_csv("dataFrame/reordering_slippage.csv", index=False)
print(f"\nResultados salvos em 'dataFrame/reordering_slippage.csv'")
df_rs.head(10)

Calculando reordering slippage para 35 blocos...
Permutações por bloco: 200

  10/35 blocos processados...
  20/35 blocos processados...
  30/35 blocos processados...

REORDERING SLIPPAGE — Pool WMATIC/USDC — Polygon
Swaps analisados          : 80
Blocos analisados         : 35

Reordering Slippage (bps):
  Média                   : -207.1094
  Mediana                 : -309.7694
  Desvio padrão           : 770.8053
  Mín                     : -1058.9191
  Máx                     : 1037.9750

Adversarial (RS < 0)      : 54  (67.5%)
Benigno    (RS >= 0)      : 26  (32.5%)

RS médio adversarial      : -682.2232 bps
RS médio benigno          : 779.6653 bps

Resultados salvos em 'dataFrame/reordering_slippage.csv'


,bloco,tx_index,preco_realizado,preco_hipotetico_medio,reordering_slippage_bps,n_swaps_no_bloco
0,87920576,112,0.091079,0.090698,-41.965484,2
1,87920576,116,0.091095,0.090922,-19.041335,2
2,87920646,78,0.090988,0.089779,-134.656844,2
3,87920646,79,0.091061,0.090374,-75.990550,2
4,87920651,1,0.091153,0.098017,700.334248,2
5,87920651,98,0.091137,0.097555,657.831794,2
6,87920920,1,0.091057,0.082978,-973.655375,3
7,87920920,52,0.091051,0.082943,-977.590923,3
8,87920920,56,0.091051,0.082900,-983.220580,3
9,87920947,3,0.091109,0.085824,-615.772201,2


## 9. Análise dos Resultados — Reordering Slippage

### O que estamos medindo?

Para entender os resultados, vale relembrar o que o Reordering Slippage representa na prática.
Dentro de cada bloco da blockchain, várias transações de swap acontecem "ao mesmo tempo" —
e quem decide a ordem em que elas são executadas é o validador do bloco. O Reordering Slippage
mede o seguinte: **o preço que você obteve foi melhor ou pior do que você obteria se a ordem
das transações fosse sorteada aleatoriamente?**

- Se o resultado for **negativo (RS < 0)**: você pagou mais caro (ou recebeu menos) do que
  receberia numa ordem aleatória. Isso é um indício de que alguém manipulou a ordem das
  transações para se beneficiar às suas custas — o que chamamos de comportamento **adversarial** ou MEV.
- Se o resultado for **positivo (RS ≥ 0)**: você se saiu melhor do que a média das ordens
  aleatórias. Isso pode acontecer por acaso ou por colisão benigna de transações, e chamamos
  de slippage **benigno**.

Os valores são expressos em **basis points (bps)**, onde 1 bps = 0,01%. Então um RS de −682 bps
significa que o usuário pagou, em média, **6,82% a mais** do que pagaria numa ordem aleatória.

---

### Visão Geral da Amostra

A análise cobriu **80 swaps** distribuídos em **35 blocos** do pool WMATIC/USDC na rede Polygon.
Apenas blocos com 2 ou mais swaps foram considerados, pois é necessário haver mais de uma
transação no bloco para que a ordem de execução faça diferença. Para cada swap, simulamos
**200 permutações aleatórias** da ordem dos trades do bloco e calculamos qual seria o preço
hipotético médio — esse valor serve de referência para medir o quanto a ordem real beneficiou
ou prejudicou o usuário.

---

### Distribuição do Reordering Slippage

| Estatística | Valor (bps) | Interpretação |
|---|---|---|
| Média | −207,11 | Em média, os swaps saíram 2,07% piores do que o esperado |
| Mediana | −309,77 | Metade dos swaps teve prejuízo acima de 3,10% |
| Desvio padrão | 770,81 | Resultados muito dispersos — há casos bem extremos |
| Mínimo | −1.058,92 | O pior swap pagou ~10,6% a mais do que deveria |
| Máximo | +1.037,98 | O melhor swap recebeu ~10,4% a mais do que deveria |

A média negativa de −207 bps já é um sinal de alerta, mas a mediana de −309 bps é ainda mais
reveladora: ela mostra que **mais da metade dos swaps** foi prejudicada de forma relevante pela
ordem de execução. O fato de a mediana ser mais negativa que a média indica que os casos
benignos, quando ocorrem, tendem a ser de grande magnitude — puxando a média para cima e
suavizando o quadro geral.

---

### Composição Adversarial vs. Benigna

| Classificação | Swaps | Proporção | RS médio |
|---|---|---|---|
| Adversarial (RS < 0) | 54 | 67,5% | −682,22 bps |
| Benigno (RS ≥ 0) | 26 | 32,5% | +779,67 bps |

**2 em cada 3 swaps** analisados apresentaram reordering slippage adversarial. Isso significa que,
na maior parte dos blocos observados, a ordem de execução das transações foi desfavorável para
os usuários — algo improvável de ocorrer sistematicamente por acaso, e que aponta para a
presença de agentes (bots de MEV) explorando ativamente a sequência das transações.

---

### Conclusão

Os resultados obtidos para o pool WMATIC/USDC na rede Polygon revelam um cenário com presença
relevante de reordenação adversarial. Com 67,5% dos swaps classificados como adversariais e um
custo médio de −682 bps para os afetados, fica evidente que mesmo em uma rede L2 — historicamente
associada a menores custos e menor exposição a MEV — os usuários não estão completamente protegidos
da extração de valor por agentes que manipulam a ordem das transações.

Vale destacar, no entanto, que esta é apenas a primeira parte da análise. A hipótese central da
pesquisa é que o reordering slippage em redes L2 como a Polygon é **significativamente menor**
do que na Ethereum Mainnet. Para validar ou refutar essa hipótese, os valores aqui obtidos
precisam ser comparados com uma coleta equivalente na rede principal. Somente com essa comparação
será possível afirmar se a Polygon de fato oferece uma experiência de trading mais justa, ou se
a extração de MEV opera em magnitudes similares independentemente da camada da rede.